# VectorCraft Scripting

Example notebook for the Python bridge to VectorCraft's headless CLI (`vectorcraft-cli`): place Plotly/SVG figures into a document and export PDF/SVG/`.vectorcraft`.

Requires VectorCraft installed (Windows default: `C:\\Program Files\\VectorCraft\\vectorcraft-cli.exe`).

In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

import sys
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from collections import defaultdict
from functools import partial
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)

import IPython
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from pyphocorehelpers.programming_helpers import IPythonHelpers
from pyphocorehelpers.assertion_helpers import Assert
from pyphocorehelpers.print_helpers import print_keys_if_possible, print_value_overview_only
from pyphocorehelpers.preferences_helpers import set_pho_preferences
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path, file_uri_from_path, sanitize_filename_for_Windows
from pyphocorehelpers.gui.Jupyter.simple_widgets import fullwidth_path_widget
from pyphocorehelpers.pho_jupyter_preview_widget.display_helpers import path_repr_with_filesystem_link

set_pho_preferences(is_concise=False)

_notebook_path: Path = Path(IPythonHelpers.try_find_notebook_filepath(IPython.extract_module_locals())).resolve()
_notebook_dir: Path = _notebook_path.parent
_package_root: Path = _notebook_dir.parent
_package_src: Path = _package_root.joinpath('src').resolve()
if str(_package_src) not in sys.path:
    sys.path.insert(0, str(_package_src))

ip = IPython.get_ipython()
ip = path_repr_with_filesystem_link(ip=ip)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
template: str = 'plotly_dark'
pio.templates.default = template

from vectorcraft_scripting import (
    CliNotFoundError,
    PlaceSpec,
    VectorCraftClient,
    VectorCraftError,
    export_plotly_figure,
    layer_geometry,
    place_into_template,
    replace_layers,
    write_plotly_svg,
)

print(f'_notebook_path: {_notebook_path}')
print(f'_package_src: {_package_src}')

## Paths / output folder

In [ ]:
outputs_folder: Path = _notebook_dir.joinpath('outputs').resolve()
outputs_folder.mkdir(parents=True, exist_ok=True)
display(fullwidth_path_widget(outputs_folder, file_name_label='VectorCraft outputs:'))
print(f'outputs_folder: {outputs_folder}')
print(f'file_uri: {file_uri_from_path(outputs_folder)}')

## Discover `vectorcraft-cli` (Windows)

In [ ]:
vc = VectorCraftClient()  # VECTORCRAFT_CLI → PATH → C:\\Program Files\\VectorCraft\\vectorcraft-cli.exe
print(f'cli_path: {vc.cli_path}')
Assert.path_exists(vc.cli_path)

catalogue: List[Dict[str, Any]] = vc.commands()
print(f'n_commands: {len(catalogue)}')
print(f'first: {catalogue[0]["id"]}')

place_docs = next((c for c in catalogue if c.get('id') == 'file.place'), None)
if place_docs is not None:
    print(f"file.place enabled={place_docs.get('enabled')} label={place_docs.get('label')!r}")
    print(str(place_docs.get('params', ''))[:500], '...')

## Smoke: `file.new` → export SVG

In [ ]:
smoke_svg: Path = outputs_folder.joinpath('smoke_blank.svg').resolve()
smoke_steps = vc.run(
    commands=[('file.new', {'width': 560, 'height': 360})],
    exports=[smoke_svg],
)
print_value_overview_only(smoke_steps)
Assert.path_exists(smoke_svg)
display(fullwidth_path_widget(smoke_svg, file_name_label='smoke SVG:'))

## Plotly figure → SVG → place → export

Creates a new document, places the chart in a rect (document points: `[x, y, width, height]` from the artboard top-left), and writes `.vectorcraft` + PDF.

In [ ]:
fig = go.Figure(go.Scatter(x=[1, 2, 3], y=[2, 1, 3], mode='lines+markers', name='demo'))
fig.update_layout(width=560, height=360, margin=dict(l=40, r=16, t=32, b=32), title='VectorCraft place demo')
fig.show()

chart_svg: Path = outputs_folder.joinpath('chart.svg').resolve()
write_plotly_svg(fig, chart_svg)
Assert.path_exists(chart_svg)

# 36 pt from left, 80 pt from top, 280×180 pt box
place = PlaceSpec(path=chart_svg, rect=[36, 80, 280, 180], crop='crop', link=False)

out_vc: Path = outputs_folder.joinpath('out.vectorcraft').resolve()
out_pdf: Path = outputs_folder.joinpath('out.pdf').resolve()

place_steps = vc.run(
    commands=[
        ('file.new', {'width': 612, 'height': 792}),  # letter-ish artboard
        place.as_command(),
    ],
    exports=[out_vc, out_pdf],
)
print_keys_if_possible('place_steps', place_steps, max_depth=2)
Assert.path_exists(out_vc)
Assert.path_exists(out_pdf)
display(fullwidth_path_widget(out_pdf, file_name_label='placed PDF:'))

## Optional: place into an existing `.vectorcraft` template

If you have a layout template, set `template_path` and run the cell below (same as the README `export_plotly_figure` / `place_into_template` helpers).

In [ ]:
template_path: Optional[Path] = None
# template_path = Path(r'H:/path/to/template.vectorcraft').resolve()

if template_path is None:
    print('Set template_path to a .vectorcraft file to run this cell.')
else:
    Assert.path_exists(template_path)
    templated_pdf: Path = outputs_folder.joinpath('from_template.pdf').resolve()
    results = export_plotly_figure(
        fig,
        chart_svg=outputs_folder.joinpath('chart_for_template.svg'),
        template=template_path,
        rect=[36, 480, 280, 180],
        exports=[templated_pdf],
        client=vc,
    )
    print_value_overview_only(results)
    display(fullwidth_path_widget(templated_pdf, file_name_label='template PDF:'))

## Inspect a document

In [ ]:
summary = vc.info(out_vc)
print_keys_if_possible('info', summary, max_depth=2)
summary.get('artboards')

## Live session (`vectorcraft --control`)

Requires the desktop app started with a control port, e.g. `vectorcraft --control 7979`. Discovery does not launch the GUI.

In [ ]:
from vectorcraft_scripting import ObjectQuery, connect_first, discover_instances, NoInstanceError, LiveVectorCraft

instances = discover_instances()
print(f'n_instances: {len(instances)}')
for inst in instances:
    print(f'  port={inst.port} pid={inst.pid}')
## END for inst in instances...

live: Optional[LiveVectorCraft] = None
try:
    live = connect_first()
    docs = live.list_documents()
    print('documents:', [d.get('title') for d in docs])
    tree = live.inspect_document(depth=2, child_limit=50)
    print_keys_if_possible('inspect_document', tree, max_depth=2)
    matches = live.find_objects(ObjectQuery(kind='Path', min_width=10, max_width=2000, limit=25))
    print(f'Path matches: {len(matches)}')
    if matches:
        print(matches[0])
except NoInstanceError as e:
    print(f'No live VectorCraft control server: {e}')
    print('Start with: & "C:\\Program Files\\VectorCraft\\vectorcraft.exe" --control 7979')
# finally:
#     if live is not None:
#         live.close()

In [ ]:
active_document = docs[0]
active_document

active_artboard = active_document['artboards'][0] ##
active_artboard

In [ ]:
# print('documents:', [d.get('title') for d in docs])
tree = live.inspect_document(depth=3, child_limit=100)
# print_keys_if_possible('inspect_document', tree, max_depth=2)

layers = tree['layers']
layers


In [ ]:
layer_names = ['kdiba_4_figure4_summary_four_bar-Laps.vectorcraft', 'ScatterOverTime_Laps_perTimeBin.svg', 'ScatterOverTime_PBEs_perTimeBin.svg', 'kdiba_4_figure4_summary_four_bar-PBEs.vectorcraft']


In [ ]:

print_keys_if_possible('layers', layers, max_depth=0)


In [ ]:

matches = live.find_objects(ObjectQuery(kind='Path', min_width=10, max_width=2000, limit=25))
print(f'Path matches: {len(matches)}')
if matches:
    print(matches[0])



In [ ]:

matches = live.find_objects(ObjectQuery(kind='Path', min_width=10, max_width=2000, limit=25))
print(f'Path matches: {len(matches)}')
if matches:
    print(matches[0])


In [ ]:
from vectorcraft_scripting import connect_first, ObjectQuery

live = connect_first()
print(live.list_documents())
print(live.find_objects(ObjectQuery(kind="Path", min_width=100)))
live.close()

In [ ]:
print()
found_layers = live.find_objects(ObjectQuery(kind="Layer"))

['Fig3a', 'Fig3b', 'fig3_c_data_diag_hist', 'C_Data']

In [ ]:
from vectorcraft_scripting.layers import Layers

docs = live.list_documents()
print('documents:', [d.get('title') for d in docs])

active_figure_name: str = 'figS3_TimeBinDurationOnDecodingPerform.pdf'
active_document = live.select_document(active_figure_name)  # title, filename, or stem
live.assert_active_document(active_figure_name)
# active_document

tree = live.inspect_document(depth=1, childLimit=500)
flat_nodes = Layers._subfn_flatten_layer_nodes(list(tree.get('layers') or []))

'Layer'
flat_nodes


In [ ]:

layers = tree['layers']
print_keys_if_possible('layers', layers, max_depth=2)
layers

In [ ]:
layers

In [ ]:
live.list_documents()

## get document with name 'Artboard 3.pdf'

## get the tree of its layers

In [ ]:
print(live.find_objects(ObjectQuery(kind="Layer", min_width=100)))

## Get specific layers

In [10]:
from vectorcraft_scripting import layer_geometry, replace_layers

# layer_names = ['kdiba_4_figure4_summary_four_bar-Laps.vectorcraft', 'ScatterOverTime_Laps_perTimeBin.svg', 'ScatterOverTime_PBEs_perTimeBin.svg', 'kdiba_4_figure4_summary_four_bar-PBEs.vectorcraft']
# layer_names = ['ScatterOverTime_Laps_perTimeBin.svg', 'Figure 4_SummaryStats_Laps_perTimeBin.svg', 'ScatterOverTime_PBEs_perTimeBin.svg', 'Figure 4_SummaryStats_PBEs_perTimeBin.svg']
# layer_names = ['Fig3a', 'Fig3b', 'fig3_c_data_diag_hist', 'C_Data']
# layer_names = ['Fig3a.svg', 'Fig3b.svg', 'C_Data.svg', 'StaticCParts', 'Fig3c_data_diag_hist.svg'] # , 'fig3_c_data_diag_hist'
layer_names = ['ScatterOverTime_Laps_perTimeBin.svg', 'ScatterOverTime_PBEs_perTimeBin.svg', 'Figure S3_SummaryStats_perTimeBin_by_time_bin_size.svg']
layer_geom_df = layer_geometry(live, layer_names)
layer_geom_df = layer_geom_df[layer_geom_df['visible']] ## only get the visible layers
layer_geom_df

parent_layers = ['Main', 'Backup']

geom_columns = ['x', 'y', 'width', 'height']  # 'name',
curr_fig_geom_dict = layer_geom_df[geom_columns].to_dict(orient='index')
## OUTPUTS: curr_fig_geom_dict
def format_layer_geom_dict(geom_dict) -> str:
    return "{\n" + "".join(
        f"    {key!r}: {value!r},\n"
        for key, value in geom_dict.items()
    ) + "}"
        


print(format_layer_geom_dict(geom_dict=curr_fig_geom_dict))

# C_Data.svg_PROGRAMMATIC	C_Data.svg	32902.0	Layer	197.47501803775822	14.034866992000815	232.8341819622417	228.0290142961035	True	False
# C_Data.svg_PROGRAMMATIC	C_Data.svg	53799.0	Layer	195.46821037898118	13.974207317621463	234.9447490297146	230.33770365816957	True	False
# curr_fig_geom_dict


active document: 'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf'  units='Inches'


,query,id,kind,x,y,width,height,visible,locked
name,,,,,,,,,
ScatterOverTime_Laps_perTimeBin.svg_PROGRAMMATIC,ScatterOverTime_Laps_perTimeBin.svg,18667,Group,2.371345,0.948557,471.350233,188.543895,True,False
ScatterOverTime_PBEs_perTimeBin.svg_PROGRAMMATIC,ScatterOverTime_PBEs_perTimeBin.svg,18934,Group,-8.432480,157.861001,484.525403,193.814069,True,False
Figure S3_SummaryStats_perTimeBin_by_time_bin_size.svg_PROGRAMMATIC,Figure S3_SummaryStats_perTimeBin_by_time_bin_...,20823,Group,11.980260,332.618103,271.765185,209.798294,True,False


{
    'ScatterOverTime_Laps_perTimeBin.svg_PROGRAMMATIC': {'x': 2.3713450469581594, 'y': 0.948557143849726, 'width': 471.3502334228814, 'height': 188.5438948423104},
    'ScatterOverTime_PBEs_perTimeBin.svg_PROGRAMMATIC': {'x': -8.432479962904514, 'y': 157.86100084746104, 'width': 484.52540347970216, 'height': 193.81406912372236},
    'Figure S3_SummaryStats_perTimeBin_by_time_bin_size.svg_PROGRAMMATIC': {'x': 11.980259810880275, 'y': 332.6181027247877, 'width': 271.7651846403536, 'height': 209.79829413492132},
}


In [ ]:
live.close()

In [ ]:
if ('live' not in globals()) or (live is None) or (not live.is_connected):
    live = connect_first()

# live.close()

In [ ]:
figs_layer_geometry = {
    'Artboard 3.pdf': {
        # 'Fig3a+b': {'x': 5.7407, 'y': 7.447218750000047, 'width': 129.1757, 'height': 241.66897968749993},
        'Fig3a.svg': {'x': 9.3114859375, 'y': 11.197580273437529, 'width': 125.6046140625, 'height': 114.23601816406247},
        'Fig3b.svg': {'x': 9.3114859375, 'y': 134.06868027343748, 'width': 125.60491406250001, 'height': 115.0475181640625},
        # 'fig3_c_data_diag_hist': {'x': 398.5329, 'y': 1.0249999999999773, 'width': 67.6481, 'height': 67.64910000000006},
        # 'C_Data.svg': {'x': 201.5388, 'y': 26.771000000000072, 'width': 242.0228, 'height': 214.85709999999995}, # #TODO 2026-10-29 16:01: - [ ] REAL C_Data
        # 'C_Data.svg': {'x': 157.8184, 'y': 8.176718749999964, 'width': 270.5089, 'height': 260.49027343750004}, #TODO 2026-10-29 16:01: - [ ] Fake C_Data from `StaticCParts`
        'C_Data.svg': {'x': 195.46821037898118, 'y': 13.974207317621463, 'width': 234.9447490297146, 'height': 230.33770365816957}, # #TODO 2026-10-30 10:54: - [ ] Modified better boundaries placement
        # 'StaticCParts': {'x': 157.8184, 'y': 8.176718749999964, 'width': 270.5089, 'height': 260.49027343750004},
        # 'Fig3c_data_diag_hist.svg': {'x': 398.5329, 'y': 1.0249999999999773, 'width': 67.6481, 'height': 67.64910000000006},
    },
    'Artboard 4.pdf': {
        'ScatterOverTime_Laps_perTimeBin.svg': {'x': 0.7921446789572979, 'y': 0.0, 'width': 388.47986937169003, 'height': 156.96156338250103},
        'Figure 4_SummaryStats_Laps_perTimeBin.svg': {'x': 369.45444366506405, 'y': 7.341050359617976, 'width': 99.67588609310917, 'height': 141.4210368639443},
        'ScatterOverTime_PBEs_perTimeBin.svg': {'x': 0.5088833995630582, 'y': 150.66224712966167, 'width': 388.47986937169, 'height': 156.961563382501},
        'Figure 4_SummaryStats_PBEs_perTimeBin.svg': {'x': 369.45444366506405, 'y': 159.06777769089888, 'width': 99.67680000000001, 'height': 141.4223335231983},
    },
    'figS3_TimeBinDurationOnDecodingPerform.pdf': {
        'ScatterOverTime_Laps_perTimeBin.svg_PROGRAMMATIC': {'x': 2.3713450469581594, 'y': 0.948557143849726, 'width': 471.3502334228814, 'height': 188.5438948423104},
        'ScatterOverTime_PBEs_perTimeBin.svg_PROGRAMMATIC': {'x': -8.432479962904514, 'y': 157.86100084746104, 'width': 484.52540347970216, 'height': 193.81406912372236},
        'Figure S3_SummaryStats_perTimeBin_by_time_bin_size.svg_PROGRAMMATIC': {'x': 11.980259810880275, 'y': 332.6181027247877, 'width': 271.7651846403536, 'height': 209.79829413492132},
    },
    'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf': {
        'ScatterOverTime_Laps_perTimeBin.svg': {'x': 0.0, 'y': 0.0, 'width': 468.27133896419633, 'height': 189.20054099563487},
        'ScatterOverTime_PBEs_perTimeBin.svg': {'x': 0.5893954560219754, 'y': 185.72098141999123, 'width': 467.69220359180815, 'height': 188.96654690578106},
        'kdiba_4_Figure S8-pre_post_delta_effect-P_Short.svg': {'x': 6.4321670348433475, 'y': 364.941057936045, 'width': 278.90913365273093, 'height': 209.8713396547812},
    },
}

# active_figure_export_folder_date: str = '2026-10-29'
active_figure_export_folder_date: str = '2026-11-03'

replacement_paths_dict = {
    'Artboard 3.pdf': Path(f'E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/2026-09-28 - EXPORTS FOR PUBLICATION/{active_figure_export_folder_date}/Figure 3 - FRI Scatter/_TEMP'),
    'Artboard 4.pdf': Path(f'E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure 4 - P_Short Scatter over Time/{active_figure_export_folder_date}'),
    'figS3_TimeBinDurationOnDecodingPerform.pdf': Path(f'E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure S3 - Effect of Time Bin Duration on Decoding Performance/Inputs/{active_figure_export_folder_date}'),
    'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf': Path(f'E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure S8 - Effect of Heuristic-filtering P_Short Scatter over Time - perEpoch/Inputs/{active_figure_export_folder_date}'),
    # 'Figure S9': Path(f'E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure S9 - Effect of Heuristic-filtering P_Short Scatter over Time - PerEpoch/Inputs/{active_figure_export_folder_date}'),
}


In [ ]:
# {'Fig3a.svg': WindowsPath('E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/2026-09-28 - EXPORTS FOR PUBLICATION/2026-10-29/Figure 3 - FRI Scatter/_TEMP/Fig3a.svg'),
#  'Fig3b.svg': WindowsPath('E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/2026-09-28 - EXPORTS FOR PUBLICATION/2026-10-29/Figure 3 - FRI Scatter/_TEMP/Fig3b.svg'),
#  'C_Data.svg': WindowsPath('E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/2026-09-28 - EXPORTS FOR PUBLICATION/2026-10-29/Figure 3 - FRI Scatter/_TEMP/C_Data.svg'),
#  'Fig3c_data_diag_hist.svg': WindowsPath('E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/2026-09-28 - EXPORTS FOR PUBLICATION/2026-10-29/Figure 3 - FRI Scatter/_TEMP/Fig3c_data_diag_hist.svg')}

In [ ]:
curr_replacement_search_path

In [8]:
if ('live' not in globals()) or (live is None) or (not live.is_connected):
    live = connect_first()

## For a particular figure
# active_figure_name: str = 'Artboard 3.pdf'# 
# active_figure_name: str = 'Artboard 4.pdf'# 
active_figure_name: str = 'figS3_TimeBinDurationOnDecodingPerform.pdf'# 
# active_figure_name: str = 'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf'

## search for files with names matching those in figs_layer_geometry in `curr_replacement_search_path`
# for active_figure_name, curr_fig_geom_dict in figs_layer_geometry.items():
curr_fig_geom_dict = figs_layer_geometry[active_figure_name]
curr_replacement_search_path: Path = replacement_paths_dict[active_figure_name]

live.select_document(active_figure_name)  # activate the matching open tab
live.assert_active_document(active_figure_name)  # fail before mutating if the wrong tab is active

backup_paths = live.backup_document(expect_title=active_figure_name)  # e.g. {'same_format': .../_BAK/Artboard 4_2026-10-29.pdf, 'vectorcraft': .../_BAK/Artboard 4_2026-10-29.vectorcraft}
print(f'backup_paths: {backup_paths}')
backup_layers_with_date: bool = False  # True: duplicate existing layers as hidden `<name>_<YYYY-MM-DD>` copies before placing
with live.undo_group(f'Replace figure layers ({datetime.now():%Y-%m-%d})'):
    replacement_df = replace_layers(live, curr_fig_geom_dict, curr_replacement_search_path, expect_title=active_figure_name, backup_layers=backup_layers_with_date)
replacement_df

{'artboards': 1,
 'author': '',
 'authorTitle': '',
 'colorMode': 'cmyk',
 'copyrightNotice': '',
 'copyrightStatus': 'unknown',
 'copyrightUrl': '',
 'created': '2026-10-02T21:51:04Z',
 'description': '',
 'keywords': [],
 'modified': '2026-11-03T18:45:21Z',
 'objects': 7396,
 'rating': 0,
 'title': 'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf',
 'units': 'Inches'}

{'artboards': 1,
 'author': '',
 'authorTitle': '',
 'colorMode': 'cmyk',
 'copyrightNotice': '',
 'copyrightStatus': 'unknown',
 'copyrightUrl': '',
 'created': '2026-10-02T21:51:04Z',
 'description': '',
 'keywords': [],
 'modified': '2026-11-03T18:45:21Z',
 'objects': 7396,
 'rating': 0,
 'title': 'Figure S8 - Effect of Heuristic-filtering perTimeBin.pdf',
 'units': 'Inches'}

backing up 'figS3_TimeBinDurationOnDecodingPerform.pdf' → 'E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure S3 - Effect of Time Bin Duration on Decoding Performance/_BAK'
  warnings for 'figS3_TimeBinDurationOnDecodingPerform_2026-11-03.pdf': ['PDF keeps the artwork but not everything a VectorCraft document holds (editable effects, symbols, swatches, styles): save as VectorCraft to keep it all editable']
  [same_format] E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure S3 - Effect of Time Bin Duration on Decoding Performance/_BAK/figS3_TimeBinDurationOnDecodingPerform_2026-11-03.pdf  (4547.2 KiB)
  [vectorcraft] E:/Dropbox (Personal)/Active/Kamran Diba Lab/Pho-Kamran-Meetings/Pho Diba Paper 2023/Figures/UneditedSources/Figure S3 - Effect of Time Bin Duration on Decoding Performance/_BAK/figS3_TimeBinDurationOnDecodingPerform_2026-11-03.vectorcraft  (5489.

H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\EXTERNAL\VectorCraft_Scripting\src\vectorcraft_scripting\layers.py:330: UserWarning: 'ScatterOverTime_Laps_perTimeBin.svg': target aspect 2.4999 (476.09x190.44 pt) != source aspect 2.4750 (388.48x156.96 pt); non-uniform scaling would be required (|log ratio| = 0.0100). Placing uniformly-scaled and centered instead: {'x': 2.3713450469581687, 'y': 0.0, 'width': 471.35023342288133, 'height': 190.4410091300099}
  warnings.warn(msg + f" Placing uniformly-scaled and centered instead: {place_rect}")


  ScatterOverTime_Laps_perTimeBin.svg: 'ScatterOverTime_Laps_perTimeBin.svg_PROGRAMMATIC' id=18667 parent='Main' backup_root=None placed_ids=[18667] deleted_ids=[]


H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\EXTERNAL\VectorCraft_Scripting\src\vectorcraft_scripting\layers.py:330: UserWarning: 'ScatterOverTime_PBEs_perTimeBin.svg': target aspect 2.1605 (484.53x224.26 pt) != source aspect 2.4750 (388.48x156.96 pt); non-uniform scaling would be required (|log ratio| = 0.1359). Placing uniformly-scaled and centered instead: {'x': -8.432479962904523, 'y': 177.88592965780703, 'width': 484.52540347970216, 'height': 195.7642115030304}
  warnings.warn(msg + f" Placing uniformly-scaled and centered instead: {place_rect}")


  ScatterOverTime_PBEs_perTimeBin.svg: 'ScatterOverTime_PBEs_perTimeBin.svg_PROGRAMMATIC' id=18934 parent='Main' backup_root=None placed_ids=[18934] deleted_ids=[]
  Figure S3_SummaryStats_perTimeBin_by_time_bin_size.svg: 'Figure S3_SummaryStats_perTimeBin_by_time_bin_size.svg_PROGRAMMATIC' id=20823 parent='Main' backup_root=None placed_ids=[20823] deleted_ids=[]


H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\EXTERNAL\VectorCraft_Scripting\src\vectorcraft_scripting\layers.py:330: UserWarning: 'Figure S3_SummaryStats_perTimeBin_by_time_bin_size.svg': target aspect 1.2670 (284.59x224.62 pt) != source aspect 1.3333 (460.80x345.60 pt); non-uniform scaling would be required (|log ratio| = 0.0510). Placing uniformly-scaled and centered instead: {'x': 3.2553841573510987, 'y': 356.97001876173067, 'width': 284.59136092010016, 'height': 213.44352069007513}
  warnings.warn(msg + f" Placing uniformly-scaled and centered instead: {place_rect}")


,replacement_file,existing_ids,source_kind,parent_id,parent_name,deleted_ids,backup_root,backup_root_id,backup_layer,backup_ids,new_layer,new_layer_id,placed_ids,source_w,source_h,aspect_ratio_error,target_x,target_y,target_w,target_h,placed_x,placed_y,placed_w,placed_h,actual_x,actual_y,actual_w,actual_h,dx,dy,dw,dh
layer,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ScatterOverTime_Laps_perTimeBin.svg,E:/Dropbox (Personal)/Active/Kamran Diba Lab/P...,[12211],Group,420,Main,[],None,None,None,[],ScatterOverTime_Laps_perTimeBin.svg_PROGRAMMATIC,18667,[18667],388.4832,156.96,0.010012,0.000000,0.000000,476.092924,190.441009,2.371345,0.000000,471.350233,190.441009,2.371345,0.948557,471.350233,188.543895,-9.325873e-15,0.948557,5.684342e-14,-1.897114
ScatterOverTime_PBEs_perTimeBin.svg,E:/Dropbox (Personal)/Active/Kamran Diba Lab/P...,[12478],Group,420,Main,[],None,None,None,[],ScatterOverTime_PBEs_perTimeBin.svg_PROGRAMMATIC,18934,[18934],388.4832,156.96,0.135902,-8.432480,163.637249,484.525403,224.261572,-8.432480,177.885930,484.525403,195.764212,-8.432480,178.861001,484.525403,193.814069,8.881784e-15,0.975071,0.000000e+00,-1.950142
Figure S3_SummaryStats_perTimeBin_by_time_bin_size.svg,E:/Dropbox (Personal)/Active/Kamran Diba Lab/P...,[16293],Group,420,Main,[],None,None,None,[],Figure S3_SummaryStats_perTimeBin_by_time_bin_...,20823,[20823],460.8000,345.60,0.051022,3.255384,351.383558,284.591361,224.616442,3.255384,356.970019,284.591361,213.443521,11.980260,360.615245,271.765185,209.798294,8.724876e+00,3.645227,-1.282618e+01,-3.645227


In [ ]:
live.is_connected

In [ ]:
live.close()